# Fixed Income Portfolio Management

## Introduction

This notebook implements a selection of optimisation models and case studies presented by Cornuejols, Pena, and Tutuncu in their textbook, Optimization Methods in Finance (2018). It also remasters a previous financial mathematics project completed at UCL. 

## Chapter 1: Asset-Liability Management

We follow the case study presented in Section 3.7 of Optimization Methods in Finance (2018).

Let $y = 2026$ denote the current year. We note that the present date is fixed as 08/27/2026 (with MM/DD/YYYY convention) for the remainder of this case study. A municipality has the following liability stream (in units of million dollars):

| 12/15/y | 6/15/y+1 | 12/15/y+1 | 6/15/y+2 | 12/15/y+2 | 6/15/y+3 | 12/15/y+3 | 6/15/y+4 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 11 | 9 | 8 | 7 | 9 | 10 | 9 | 12 |

| 12/15/y+4 | 6/15/y+5 | 12/15/y+5 | 6/15/y+6 | 12/15/y+6 | 6/15/y+7 | 12/15/y+7 | 6/15/y+8 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 9 | 6 | 5 | 7 | 9 | 7 | 8 | 7 |

### 1.1 Term structure of Treasury rates and liability attributes

The first task is to value this stream off the current Treasury curve and to record its interest-rate sensitivity. We take the U.S. Treasury constant-maturity par yields published on FRED (the `DGS*` series, which reproduce the H.15 / Treasury CMT curve).

These yields are quoted as semi-annual bond-equivalent rates, not as annually or continuously compounded zeros. A figure such as $4.20\%$ on the two-year point means a par coupon of $4.20\%$ paid twice a year. The matching discount factor at maturity $t$ (measured in ACT/365.25 years from 27 August 2026) is therefore

$$
DF(t)=\bigl(1+y(t)/2\bigr)^{-2t}.
$$

The liability dates do not sit on the published tenors (1M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y). We interpolate $y(t)$ linearly between those knots and, for this section, treat the interpolated CMT as the rate used to discount that cash flow. That is a standard first pass; a bootstrap from par yields to spots would be the natural refinement.

Writing $C_i$ for the payment at date $T_i$, the three liability attributes we need later in the immunization model are

$$
PV=\sum_i C_i\,DF_i,
\qquad
D_{\$}=\sum_i \frac{t_i\,C_i\,DF_i}{1+y_i/2},
\qquad
C_{\$}=\sum_i \frac{t_i(t_i+1/2)\,C_i\,DF_i}{(1+y_i/2)^2}.
$$

Dollar duration is the first derivative of present value with respect to yield (dollars per unit of yield); DV01 is that quantity times $10^{-4}$. Dollar convexity is the second derivative. On the 27 August 2026 curve this stream is worth **\$113.17 million**, with dollar duration **398.03** and dollar convexity **2,154**. Macaulay duration is about 3.59 years.

The rest of the chapter treats these three numbers as fixed liability-side inputs and chooses a portfolio of Treasuries to match them.

In [1]:
import pandas_datareader.data as web
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. curve (your code)
# ------------------------------------------------------------------
series = ["DGS1MO", "DGS3MO", "DGS6MO", "DGS1",
          "DGS2", "DGS3", "DGS5", "DGS7", "DGS10", "DGS20", "DGS30"]

rates = web.DataReader(series, "fred", start="2026-08-20", end="2026-08-28")
curve = rates.loc[: "2026-08-28"].dropna(how="all").iloc[-1]

print("Curve date:", curve.name.date())
print(curve)

VAL_DATE = pd.Timestamp(curve.name)   # 2026-08-27
Y = VAL_DATE.year                     # 2026

# ------------------------------------------------------------------
# 2. map FRED IDs -> maturity in years, yields to decimals
# ------------------------------------------------------------------
tenor_years = {
    "DGS1MO": 1/12, "DGS3MO": 0.25, "DGS6MO": 0.50, "DGS1": 1.0,
    "DGS2": 2.0,    "DGS3": 3.0,    "DGS5": 5.0,    "DGS7": 7.0,
    "DGS10": 10.0,  "DGS20": 20.0,  "DGS30": 30.0,
}
knots_t = np.array([tenor_years[s] for s in curve.index])
knots_y = curve.values.astype(float) / 100.0   # 4.20 -> 0.0420

def y_linear(t):
    """Linear interp of CMT par BEY. Flat outside the first/last knot."""
    return float(np.interp(t, knots_t, knots_y))

# ------------------------------------------------------------------
# 3. liability schedule, $ millions, y = curve year
# ------------------------------------------------------------------
payments = pd.DataFrame([
    (f"{Y}-12-15",   11),
    (f"{Y+1}-06-15",  9),
    (f"{Y+1}-12-15",  8),
    (f"{Y+2}-06-15",  7),
    (f"{Y+2}-12-15",  9),
    (f"{Y+3}-06-15", 10),
    (f"{Y+3}-12-15",  9),
    (f"{Y+4}-06-15", 12),
    (f"{Y+4}-12-15",  9),
    (f"{Y+5}-06-15",  6),
    (f"{Y+5}-12-15",  5),
    (f"{Y+6}-06-15",  7),
    (f"{Y+6}-12-15",  9),
    (f"{Y+7}-06-15",  7),
    (f"{Y+7}-12-15",  8),
    (f"{Y+8}-06-15",  7),
], columns=["date", "cf"])
payments["date"] = pd.to_datetime(payments["date"])

# ------------------------------------------------------------------
# 4. discount each flow  (CMT treated as spot BEY — linear method)
# ------------------------------------------------------------------
payments["t"]  = (payments["date"] - VAL_DATE).dt.days / 365.25
payments["y"]  = payments["t"].map(y_linear)
payments["df"] = 1.0 / (1.0 + payments["y"] / 2.0) ** (2.0 * payments["t"])
payments["pv"] = payments["cf"] * payments["df"]

# dollar duration piece: -dPV_i/dy_i  for a 1.00 move in that yield
# d/dy [ (1+y/2)^(-2t) ] = -t * DF / (1+y/2)
payments["dol_dur_i"] = payments["t"] * payments["pv"] / (1.0 + payments["y"] / 2.0)

# dollar convexity piece: d²PV_i/dy_i²
# t(t + 1/2) * CF * DF / (1+y/2)^2
payments["dol_conv_i"] = (
    payments["pv"] * payments["t"] * (payments["t"] + 0.5)
    / (1.0 + payments["y"] / 2.0) ** 2
)

PV        = payments["pv"].sum()
DOL_DUR   = payments["dol_dur_i"].sum()   # $mn per 1.00 (100%) yield
DV01      = DOL_DUR * 1e-4                # $mn per 1 bp
DOL_CONV  = payments["dol_conv_i"].sum()  # $mn per (yield)^2
MAC       = (payments["t"] * payments["pv"]).sum() / PV
MOD       = DOL_DUR / PV                  # weighted modified duration

show = payments.copy()
show["y"] = show["y"] * 100
print("\nPer-payment table")
print(show[["date", "t", "cf", "y", "df", "pv", "dol_dur_i", "dol_conv_i"]]
      .round(6).to_string(index=False))

print("\n---------------- results ----------------")
print(f"Valuation date          {VAL_DATE.date()}")
print(f"PV of liabilities       {PV:,.4f}  $ million")
print(f"Macaulay duration       {MAC:.4f}  years")
print(f"Modified duration       {MOD:.4f}")
print(f"Dollar duration         {DOL_DUR:,.4f}  $mn per 1.00 yield")
print(f"DV01                    {DV01:,.6f}  $mn per bp")
print(f"Dollar convexity        {DOL_CONV:,.4f}  $mn per (yield)^2")

Curve date: 2026-08-28
DGS1MO    3.84
DGS3MO    3.90
DGS6MO    4.02
DGS1      4.15
DGS2      4.34
DGS3      4.41
DGS5      4.48
DGS7      4.59
DGS10     4.73
DGS20     5.21
DGS30     5.22
Name: 2026-08-28 00:00:00, dtype: float64

Per-payment table
      date        t  cf        y       df        pv  dol_dur_i  dol_conv_i
2026-12-15 0.298426  11 3.923244 0.988472 10.873196   3.182414    2.492037
2027-06-15 0.796715   9 4.097146 0.968204  8.713834   6.803073    8.644554
2027-12-15 1.297741   8 4.206571 0.947409  7.579273   9.633319   16.961468
2028-06-15 1.798768   7 4.301766 0.926299  6.484092  11.417793   25.694206
2028-12-15 2.299795   9 4.360986 0.905547  8.149925  18.343181   50.261199
2029-06-15 2.798084  10 4.395866 0.885443  8.854425  24.242585   78.234527
2029-12-15 3.299110   9 4.420469 0.865673  7.791059  25.147737   93.473050
2030-06-15 3.797399  12 4.437909 0.846468 10.157619  37.735206  158.643022
2030-12-15 4.298426   9 4.455445 0.827446  7.447010  31.312856  146.978151
2

### 1.2 Fixed Income Universe

The dedicated-portfolio problem needs a set of default-free, non-callable instruments whose cash flows can be lined up against the municipality’s dates. We take the full Barron’s screen of U.S. Treasury bills, notes and bonds as of Friday 28 August 2026 (Tullett Prebon, 3pm Eastern). That is one session after the 27 August curve used in §1.1; we treat the mismatch as a data limitation rather than mixing two pricing dates inside the same model.

Note and bond quotes arrive in the usual 32nds convention (99.31 is $99 + 31/32$), so they are converted to decimal prices per 100 of par. Bill quotes are bank-discount rates, not prices; we convert the asked discount to a price with $P = 100\,(1 - d \cdot t / 360)$. Throughout we assume purchases at the ask. TIPS, FRNs and any residual callables are left out. Issues that have already matured by 27 August 2026 are dropped.

In [2]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# 1.2 Universe of Treasury bills, notes and bonds
# ------------------------------------------------------------------
SETTLE = pd.Timestamp("2026-08-28")   # Barron's quote date
VAL    = pd.Timestamp("2026-08-27")   # §1.1 valuation date
FACE   = 100.0

NOTES_CSV = "bondUniverse08282026_barrons.csv"
BILLS_CSV = "bondUniverse08282026_barrons_bills.csv"

def from_32nds(x):
    """Barron's 99.31 -> 99 + 31/32.  99.32 -> 100."""
    if pd.isna(x):
        return np.nan
    x = float(x)
    whole = np.floor(x)
    ticks = np.round((x - whole) * 100.0)
    return float(whole + ticks / 32.0)


# ----- notes & bonds -----
nb = pd.read_csv(NOTES_CSV)
nb = nb.dropna(subset=["Maturity", "Asked"]).copy()
nb["maturity"] = pd.to_datetime(nb["Maturity"])
nb["coupon"]   = pd.to_numeric(nb["Coupon"], errors="coerce") / 100.0
nb["bid"]      = nb["Bid"].map(from_32nds)
nb["ask"]      = nb["Asked"].map(from_32nds)
nb["ytm"]      = pd.to_numeric(nb["Asked yield"], errors="coerce") / 100.0
nb["price"]    = nb["ask"]
nb["ttm_days"] = (nb["maturity"] - SETTLE).dt.days
nb["type"]     = np.where(nb["ttm_days"] > 10 * 365, "bond", "note")


# ----- bills: Bid/Asked are bank-discount percents -----
bl = pd.read_csv(BILLS_CSV)
bl = bl[bl["Maturity"].astype(str).str.contains(r"\d", na=False)].copy()
bl["maturity"] = pd.to_datetime(bl["Maturity"])
days = (bl["maturity"] - SETTLE).dt.days.clip(lower=1)
d_bid = pd.to_numeric(bl["Bid"], errors="coerce") / 100.0
d_ask = pd.to_numeric(bl["Asked"], errors="coerce") / 100.0
bl["bid"]    = FACE * (1.0 - d_bid * days / 360.0)
bl["ask"]    = FACE * (1.0 - d_ask * days / 360.0)
bl["ytm"]    = pd.to_numeric(bl["Asked yield"], errors="coerce") / 100.0
bl["coupon"] = 0.0
bl["price"]  = bl["ask"]
bl["type"]   = "bill"
bl["ttm_days"] = days


# ----- single book -----
cols = ["type", "maturity", "coupon", "bid", "ask", "price", "ytm"]
univ = pd.concat([nb[cols], bl[cols]], ignore_index=True)
univ = univ.loc[univ["maturity"] > VAL].copy()
univ = univ.sort_values(["maturity", "coupon"]).reset_index(drop=True)
univ["ttm_years"] = (univ["maturity"] - VAL).dt.days / 365.25
univ["id"] = (
    univ["type"].str[0].str.upper()
    + univ["maturity"].dt.strftime("%Y%m%d")
    + "_"
    + (univ["coupon"] * 1000).round().astype(int).astype(str)
)

univ.to_csv("ust_universe_20260828.csv", index=False)


# ----- snapshot for the write-up -----
def bucket(row):
    if row["type"] == "bill":
        return "Bills, Sep 2026 – Aug 2027"
    if row["maturity"] <= pd.Timestamp("2034-06-15"):
        return "Notes/bonds through Jun 2034"
    return "Bonds 2035–2056"

snap = univ.copy()
snap["bucket"] = snap.apply(bucket, axis=1)
g = snap.groupby("bucket", sort=False)

summary = pd.DataFrame({
    "n":            g.size(),
    "first_mat":    g["maturity"].min().dt.strftime("%Y-%m-%d"),
    "last_mat":     g["maturity"].max().dt.strftime("%Y-%m-%d"),
    "ask_min":      g["ask"].min().round(2),
    "ask_max":      g["ask"].max().round(2),
    "coupon_min_%": (g["coupon"].min() * 100).round(2),
    "coupon_max_%": (g["coupon"].max() * 100).round(2),
    "ytm_min_%":    (g["ytm"].min() * 100).round(2),
    "ytm_max_%":    (g["ytm"].max() * 100).round(2),
})

print("Counts by type")
print(univ["type"].value_counts().to_string())
print(f"\nN = {len(univ)}   {univ.maturity.min().date()} → {univ.maturity.max().date()}")
print("\nSnapshot")
print(summary.to_string())
print("\nHead")
print(univ.head().to_string(index=False))

Counts by type
type
note    253
bond    101
bill     50

N = 404   2026-08-31 → 2056-08-15

Snapshot
                                n   first_mat    last_mat  ask_min  ask_max  coupon_min_%  coupon_max_%  ytm_min_%  ytm_max_%
bucket                                                                                                                       
Notes/bonds through Jun 2034  243  2026-08-31  2034-05-15    85.59   106.12          0.38          6.63       0.72       4.62
Bills, Sep 2026 – Aug 2027     50  2026-09-01  2027-08-05    96.24    99.96          0.00          0.00       3.66       4.12
Bonds 2035–2056               111  2034-08-15  2056-08-15    45.69   102.69          1.13          5.13       4.61       5.31

Head
type   maturity  coupon       bid        ask      price    ytm  ttm_years           id
note 2026-08-31  0.0075 99.968750 100.000000 100.000000 0.0072   0.010951  N20260831_8
note 2026-08-31  0.0138 99.968750 100.000000 100.000000 0.0134   0.010951 N20260831_14
no

The cleaned book has 404 lines — 50 bills, 253 notes and 101 bonds — running from 31 August 2026 to 15 August 2056. We see a compact summary of the file above. Short bills cover the first liability date in December 2026; coupon notes are dense through the first half of 2034, which is the last payment on the liability stream; a thinner tail of long bonds remains if we later want extra convexity. The optimiser in the next section is free to use the whole list and will typically hold only a sparse subset.

### 1.3 Dedication/Cash-Flow Matching Optimisation

Section 3.7 asks for the cheapest portfolio from the §1.2 universe that meets the municipality’s dates exactly, with two extra restrictions: surplus cash rolled from one date to the next earns nothing, and short sales are forbidden. That is the dedication (cash-matching) linear program in §3.1 of the book.
Let $x_j$ be the face amount of bond $j$ in millions of dollars — not a count of “100 lots.” An ask of 100.47 means you pay $100.47/100$ dollars today per dollar of par; you can take as much face as the constraints require. $p_j$ is that asked price as a fraction of par. $F_{tj}$ is the cash, per dollar of face, that bond $j$ makes available on liability date $t$: coupons or principal falling between two grid dates are parked until the next $L_t$, again at 0%. Surplus after paying $L_t$ is $s_t\ge 0$. The programme is
$$\begin{align*}
\min\quad &\mathbf{p}^{\top}\mathbf{x} \\
\text{s.t.}\quad
&\sum_j F_{1j}x_j - s_1 = L_1, \\
&\sum_j F_{tj}x_j + s_{t-1} - s_t = L_t, \qquad t=2,\ldots,16, \\
&x_j\ge 0,\qquad s_t\ge 0.
\end{align*}$$
The objective is today’s invoice at Barron’s asks. The equalities force every liability to be paid on the day it is due. Because leftover cash does not grow, there is no reason to warehouse money between dates.
We solve it in CVXPY with Clarabel. The problem is feasible. After discarding interior-point crumbs below $1,000 of face (Clarabel approaches the boundary through the interior, so “not held” comes back as $10^{-7}$ rather than a hard zero), the support is sixteen notes, one sitting immediately before each payment date. Every surplus $s_t$ is zero. The cost of that book is $111.61 million. A simplex solver (CBC) finds the same 16 names and the same objective with exact zeros, which is why we treat the extra $0.0002$ tickets as numerical dust rather than a seventeenth holding.
The $111.61 million is not the same object as the $113.17 million CMT present value in §1.1. One is a model discount of the liability stream; the other is a market outlay for a cash-matched portfolio.
The shape is what cash matching is supposed to look like. With many issues near 15 June / 15 December (or the month-end just before), the cheapest way to hit $L_t$ is to buy a redemption in that bucket and scale the face so that principal, plus any coupons landing there from this bond and from longer notes still alive, equals $L_t$. Face is therefore not simply $L_t$ divided by the quoted price. Long 20- and 30-year bonds never enter: principal after June 2034 cannot pay a liability, and their coupons are an expensive substitute for a nearer note. Bills are likewise unused.
So: lowest-cost dedicated portfolio $111.61mn, sixteen notes, no bills, no longs, surplus identically zero. That book is the benchmark if we later compare dedication with immunization.

In [3]:
import calendar
import numpy as np
import pandas as pd
import cvxpy as cp

VAL  = pd.Timestamp("2026-08-27")
Y    = 2026
FACE = 100.0

liab = pd.DataFrame([
    (f"{Y}-12-15", 11), (f"{Y+1}-06-15", 9), (f"{Y+1}-12-15", 8), (f"{Y+2}-06-15", 7),
    (f"{Y+2}-12-15", 9), (f"{Y+3}-06-15", 10),(f"{Y+3}-12-15", 9), (f"{Y+4}-06-15", 12),
    (f"{Y+4}-12-15", 9), (f"{Y+5}-06-15", 6), (f"{Y+5}-12-15", 5), (f"{Y+6}-06-15", 7),
    (f"{Y+6}-12-15", 9), (f"{Y+7}-06-15", 7), (f"{Y+7}-12-15", 8), (f"{Y+8}-06-15", 7),
], columns=["date", "L"])
liab["date"] = pd.to_datetime(liab["date"])
grid = list(liab["date"])
Lvec = liab["L"].to_numpy(float)
m, n = len(grid), len(univ)


def coupon_schedule(maturity, coupon, settle):
    """(date, $ per $1 of face) after settle, principal included."""
    y, mo, d = int(maturity.year), int(maturity.month), int(maturity.day)
    dt = pd.Timestamp(maturity)
    dates = []
    while dt > settle:
        dates.append(dt)
        mo -= 6
        if mo <= 0:
            mo += 12
            y -= 1
        last = calendar.monthrange(y, mo)[1]
        dt = pd.Timestamp(year=y, month=mo, day=min(d, last))
    out = []
    for T in sorted(dates):
        if float(coupon) == 0.0:          # bill
            if T == pd.Timestamp(maturity):
                out.append((T, 1.0))
            continue
        amt = float(coupon) / 2.0
        if T == pd.Timestamp(maturity):
            amt += 1.0
        out.append((T, amt))
    return out


def next_liability(t):
    for k, g in enumerate(grid):
        if t <= g:
            return k
    return None


F = np.zeros((m, n))
for j, row in univ.iterrows():
    for T, amt in coupon_schedule(row["maturity"], row["coupon"], VAL):
        k = next_liability(T)
        if k is not None:
            F[k, j] += amt

p = univ["price"].to_numpy(float) / FACE     # $ paid today per $1 face

x = cp.Variable(n, nonneg=True)              # face, $mn
s = cp.Variable(m, nonneg=True)              # surplus carried at 0%

cons = [F[0] @ x - s[0] == Lvec[0]]
for t in range(1, m):
    cons += [F[t] @ x + s[t - 1] - s[t] == Lvec[t]]

prob = cp.Problem(cp.Minimize(p @ x), cons)
prob.solve(solver=cp.CLARABEL, verbose=False)
# fallbacks if Clarabel is missing: cp.SCIPY, cp.OSQP, cp.HIGHS, cp.ECOS

print("status:", prob.status)
print("portfolio cost ($mn):", round(float(prob.value), 4))

xv = np.asarray(x.value).ravel()
sv = np.asarray(s.value).ravel()
tol = 1e-3

rows = []
for j in np.where(xv > tol)[0]:
    rows.append({
        "id":       univ.loc[j, "id"],
        "type":     univ.loc[j, "type"],
        "maturity": univ.loc[j, "maturity"].date(),
        "coupon_%": round(float(univ.loc[j, "coupon"]) * 100, 3),
        "ask":      round(float(univ.loc[j, "price"]), 4),
        "face_$mn": round(float(xv[j]), 4),
        "cost_$mn": round(float(p[j] * xv[j]), 4),
    })
held = pd.DataFrame(rows).sort_values("maturity")
print(held.to_string(index=False))
print("n holdings:", len(held))
print("surplus path ($mn):", np.round(np.maximum(sv, 0), 4).tolist())

status: optimal
portfolio cost ($mn): 111.6088
          id type   maturity  coupon_%      ask  face_$mn  cost_$mn
N20261115_65 note 2026-11-15      6.50 100.4688    8.2978    8.3367
N20270615_46 note 2027-06-15      4.63 100.4062    6.5673    6.5940
N20271115_61 note 2027-11-15      6.13 102.1562    5.7196    5.8429
N20280615_39 note 2028-06-15      3.88  99.2188    4.8948    4.8566
N20281115_52 note 2028-11-15      5.25 101.8438    6.9898    7.1187
N20290430_46 note 2029-04-30      4.63 100.5938    8.1733    8.2218
N20291130_41 note 2029-11-30      4.13  99.0938    7.3625    7.2958
N20300515_62 note 2030-05-15      6.25 106.1250   10.5145   11.1586
N20301031_49 note 2030-10-31      4.88 101.5938    7.8431    7.9681
N20310531_46 note 2031-05-31      4.63 100.5938    5.0345    5.0644
N20311130_41 note 2031-11-30      4.13  98.2500    4.1510    4.0784
N20320531_41 note 2032-05-31      4.13  98.0000    6.2368    6.1120
N20321115_41 note 2032-11-15      4.13  97.7188    8.3656    8.1747
N

### 1.4 Parallel Shift Hedging (Immunisation)

Dedication in §1.3 paid for a dollar on the day it was due. That is safe and expensive: sixteen notes, surplus identically zero, invoice 111.61 million dollars. Question 5 asks for a cheaper book that only matches three *summary* statistics of the same liability stream — present value, dollar duration and dollar convexity — and forbids short sales. That is immunisation as in (3.1) of the book.

A word on the three numbers, because they are doing all the work. Present value is the CMT discount of every remaining cash flow, the 113.17 million dollars from §1.1. Dollar duration is the first derivative of that present value with respect to a parallel move in yield,

$$
DD = -\frac{\partial PV}{\partial y} = \sum_i \frac{t_i\, C_i\, DF_i}{1 + y_i/2}.
$$

On the liability stream it is 398.03 (about 39.8 thousand dollars per basis point). Matching it means that, to first order, assets and liabilities reprice by the same dollar amount when every CMT yield ticks up or down together. Dollar convexity is the second derivative,

$$
DC = \frac{\partial^2 PV}{\partial y^2} = \sum_i \frac{t_i(t_i + 1/2)\, C_i\, DF_i}{(1 + y_i/2)^2}.
$$

Liabilities have dollar convexity $DC_L = 2154$. Positive net convexity ($DC_A \ge DC_L$) is favourable: for a large parallel shift the asset book gains more, or loses less, than a duration-matched linear approximation would say. Hence the last constraint is an inequality.

Each bond $j$ gets its own $PV_j$, $DD_j$ and $DC_j$ from its coupon and principal dates, using the same semi-annual BEY discounting as §1.1. $x_j$ is again face in millions of dollars; an ask of 100.06 means you pay $100.06/100$ today per dollar of par. The programme is

$$
\begin{align*}
\min \quad & \mathbf{p}^{\top} \mathbf{x} \\
\text{s.t.} \quad
& \sum_j PV_j x_j = PV_L, \\
& \sum_j DD_j x_j = DD_L, \\
& \sum_j DC_j x_j \ge DC_L, \\
& x_j \ge 0.
\end{align*}
$$

The objective is the Barron’s ask invoice, so the cost lines up against the dedicated 111.61 million dollars. We solve it in CVXPY. The problem is feasible. Only two equalities bind — convexity comes out slack — so a basic solution uses **two** names, not sixteen. The cheapest pair on the 404-issue screen is a barbell:

| id | maturity | coupon | ask | face (mn) | cost (mn) |
|---|---|---:|---:|---:|---:|
| N20280831_44 | 31 Aug 2028 | 4.38% | 100.06 | 91.73 | 91.79 |
| B20440515_46 | 15 May 2044 | 4.63% | 93.69 | 19.61 | 18.37 |
| **Total** | | | | | **110.16** |

Model PV is 113.17 and dollar duration is 398.03, both exact. Portfolio convexity is 3838, well above 2154: the 2044 leg overshoots convexity while helping duration, which is why the inequality does not bind. The invoice is **110.16 million dollars**, **1.45 million dollars** less than dedication. That saving is the point of dropping cash-flow matching. The solver searched every bill, note and bond and kept the two that delivered the three moments at the lowest ask; the other 402 lost on price.

Two bonds feels thin, and it is. An LP with two binding equalities will not sprinkle leftover face across extra notes just to look diversified — that would raise the cost. Duration of the liabilities is about 3.6 years; a ~2-year note and an ~18-year bond are the usual barbell that averages to that number. Concentration is the price of the 1.45 million. Unlike §1.3, nothing here forces a redemption onto 15 December 2026. If yields do not move, you may have to sell or roll well before a liability date.

The hedge is only a **small parallel** move in the CMT used to compute the three moments. A steepener, flattener or butterfly shifts the 2028 and 2044 points differently from the 16-date liability strip, and extra convexity does not span those twist factors. Dedication still wins if the municipality cannot miss a payment date. Immunisation wins if the mandate is “match value and first-order parallel risk as cheaply as possible.” Key-rate or PCA constraints would thicken the book without going back to full cash matching; they are not part of (3.1).

In [4]:
import calendar
import numpy as np
import pandas as pd
import cvxpy as cp

VAL  = pd.Timestamp("2026-08-27")
Y    = 2026
FACE = 100.0

knots_t = np.array([1/12, 0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])
knots_y = np.array([3.81, 3.84, 3.94, 4.04, 4.20, 4.30,
                    4.38, 4.52, 4.67, 5.18, 5.19]) / 100.0

def y_lin(t):
    return np.interp(np.asarray(t, float), knots_t, knots_y)

def metrics_of_cfs(times, cfs):
    times = np.asarray(times, float)
    cfs   = np.asarray(cfs, float)
    yt = y_lin(times)
    df = (1 + yt/2.0) ** (-2.0 * times)
    pv = cfs * df
    dd = times * pv / (1 + yt/2.0)
    dc = pv * times * (times + 0.5) / (1 + yt/2.0)**2
    return float(pv.sum()), float(dd.sum()), float(dc.sum())

def coupon_schedule(maturity, coupon, settle):
    y, mo, d = int(maturity.year), int(maturity.month), int(maturity.day)
    dt = pd.Timestamp(maturity)
    dates = []
    while dt > settle:
        dates.append(dt)
        mo -= 6
        if mo <= 0:
            mo += 12
            y -= 1
        last = calendar.monthrange(y, mo)[1]
        dt = pd.Timestamp(year=y, month=mo, day=min(d, last))
    out = []
    for T in sorted(dates):
        if float(coupon) == 0.0:
            if T == pd.Timestamp(maturity):
                out.append((T, 1.0))
            continue
        amt = float(coupon) / 2.0
        if T == pd.Timestamp(maturity):
            amt += 1.0
        out.append((T, amt))
    return out

# --- liability attributes (same convention as §1.1) ---
liab = pd.DataFrame({
    "date": pd.to_datetime([
        f"{Y}-12-15", f"{Y+1}-06-15", f"{Y+1}-12-15", f"{Y+2}-06-15",
        f"{Y+2}-12-15", f"{Y+3}-06-15", f"{Y+3}-12-15", f"{Y+4}-06-15",
        f"{Y+4}-12-15", f"{Y+5}-06-15", f"{Y+5}-12-15", f"{Y+6}-06-15",
        f"{Y+6}-12-15", f"{Y+7}-06-15", f"{Y+7}-12-15", f"{Y+8}-06-15",
    ]),
    "cf": [11, 9, 8, 7, 9, 10, 9, 12, 9, 6, 5, 7, 9, 7, 8, 7],
})
lt = (liab["date"] - VAL).dt.days.to_numpy() / 365.25
PV_L, DD_L, DC_L = metrics_of_cfs(lt, liab["cf"].to_numpy())
print(f"Liabilities  PV={PV_L:.4f}  DD={DD_L:.4f}  DC={DC_L:.4f}")

# --- per $1 of face ---
n = len(univ)
PV = np.zeros(n); DD = np.zeros(n); DC = np.zeros(n)
for j, row in univ.iterrows():
    sched = coupon_schedule(row["maturity"], row["coupon"], VAL)
    if not sched:
        continue
    ts = np.array([(T - VAL).days / 365.25 for T, _ in sched])
    cs = np.array([a for _, a in sched])
    PV[j], DD[j], DC[j] = metrics_of_cfs(ts, cs)

p = univ["price"].to_numpy(float) / FACE     # ask invoice per $1 face

x = cp.Variable(n, nonneg=True)
cons = [
    PV @ x == PV_L,
    DD @ x == DD_L,
    DC @ x >= DC_L,
]
prob = cp.Problem(cp.Minimize(p @ x), cons)
prob.solve(solver=cp.CLARABEL, verbose=False)

print("status:", prob.status)
print("immunization cost ($mn):", round(float(prob.value), 4))

DED_COST = 111.6088
print("saving vs dedication ($mn):", round(DED_COST - float(prob.value), 4))

xv = np.asarray(x.value).ravel()
tol = 1e-3
idx = np.where(xv > tol)[0]
held = univ.loc[idx].copy()
held["face_$mn"] = xv[idx]
held["cost_$mn"] = p[idx] * xv[idx]
held = held.sort_values("maturity")
print(held[["id", "type", "maturity", "coupon", "price", "face_$mn", "cost_$mn"]].to_string(index=False))
print("portfolio PV, DD, DC:",
      round(float(PV @ xv), 4),
      round(float(DD @ xv), 4),
      round(float(DC @ xv), 4))

Liabilities  PV=113.1725  DD=398.0317  DC=2154.2431
status: optimal
immunization cost ($mn): 110.1578
saving vs dedication ($mn): 1.451
          id type   maturity  coupon    price  face_$mn  cost_$mn
N20280831_44 note 2028-08-31  0.0438 100.0625 91.732417 91.789749
B20440515_46 bond 2044-05-15  0.0463  93.6875 19.605626 18.368021
portfolio PV, DD, DC: 113.1725 398.0317 3837.7842


### 1.5 Hybrid: cash-match the first three years, immunise the tail

As proposed, we adopt a hybrid approach. Dates through 15 June 2029 sit inside three years of 27 August 2026; those six payments are cash-matched exactly as in §1.3. The remaining ten payments (15 December 2029 through 15 June 2034) are treated as a tail liability and matched only in present value, dollar duration and dollar convexity, as in §1.4.

Write $K=6$ for the last cash-match date. Bond $j$ is split at that cutoff. Cash on or before 15 June 2029 goes into the early matrix $F^{\mathrm{early}}$ and the surplus chain. Cash after that date builds tail attributes $PV_j^{\mathrm{tail}}$, $DD_j^{\mathrm{tail}}$, $DC_j^{\mathrm{tail}}$ off the same CMT as §1.1. The tail liabilities themselves are

$$
PV_L^{\mathrm{tail}}=62.44,\qquad DD_L^{\mathrm{tail}}=324.06,\qquad DC_L^{\mathrm{tail}}=1970.81.
$$

The programme is

$$
\begin{align*}
\min \quad & \mathbf{p}^{\top}\mathbf{x} \\
\text{s.t.} \quad
& \sum_j F^{\mathrm{early}}_{1j}x_j-s_1=L_1, \\
& \sum_j F^{\mathrm{early}}_{tj}x_j+s_{t-1}-s_t=L_t,\qquad t=2,\ldots,6, \\
& \sum_j PV_j^{\mathrm{tail}}x_j=PV_L^{\mathrm{tail}}, \\
& \sum_j DD_j^{\mathrm{tail}}x_j=DD_L^{\mathrm{tail}}, \\
& \sum_j DC_j^{\mathrm{tail}}x_j\ge DC_L^{\mathrm{tail}}, \\
& x_j\ge 0,\qquad s_t\ge 0.
\end{align*}
$$

Coupons from the tail barbell that fall in 2026–29 help the early equalities; that is intended. Surplus is not carried from June 2029 into the tail at 0 percent — that would be silent extra dedication.

| Strategy | Cost (mn) | vs dedication |
|---|---:|---:|
| Dedication, §1.3 | 111.61 | — |
| Hybrid, Q6 | **110.81** | save 0.80 |
| Immunisation, §1.4 | 110.16 | save 1.45 |

The hybrid book is eight names: the same style of near-date notes used in §1.3 for the six early liabilities, then a 31 August 2030 note plus the 15 May 2044 4.63s for the tail.

| id | maturity | coupon | ask | face (mn) | cost (mn) |
|---|---|---:|---:|---:|---:|
| N20261115_65 | 15 Nov 2026 | 6.50% | 100.47 | 8.44 | 8.47 |
| N20270615_46 | 15 Jun 2027 | 4.63% | 100.41 | 6.71 | 6.74 |
| N20271115_61 | 15 Nov 2027 | 6.13% | 102.16 | 5.86 | 5.99 |
| N20280615_39 | 15 Jun 2028 | 3.88% | 99.22 | 5.04 | 5.00 |
| N20281115_52 | 15 Nov 2028 | 5.25% | 101.84 | 7.14 | 7.27 |
| N20290430_46 | 30 Apr 2029 | 4.63% | 100.59 | 8.33 | 8.38 |
| N20300831_41 | 31 Aug 2030 | 4.13% | 98.81 | 60.03 | 59.32 |
| B20440515_46 | 15 May 2044 | 4.63% | 93.69 | 10.28 | 9.63 |
| **Total** | | | | | **110.81** |

Early surplus is zero. Tail PV and dollar duration match exactly; tail convexity is 2703 against 1971, so the inequality is slack again. The 0.80 million saving versus dedication is the tail no longer being cash-matched. The 0.65 million extra versus full immunisation is the price of not missing a payment in 2026–29. The tail barbell is still only a parallel-shift hedge.



In [5]:
import calendar
import numpy as np
import pandas as pd
import cvxpy as cp

VAL  = pd.Timestamp("2026-08-27")
Y    = 2026
FACE = 100.0
CUT  = pd.Timestamp("2029-06-15")   # last cash-match date

knots_t = np.array([1/12, 0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])
knots_y = np.array([3.81, 3.84, 3.94, 4.04, 4.20, 4.30,
                    4.38, 4.52, 4.67, 5.18, 5.19]) / 100.0

def y_lin(t):
    return np.interp(np.asarray(t, float), knots_t, knots_y)

def metrics_of_cfs(times, cfs):
    times = np.asarray(times, float)
    cfs   = np.asarray(cfs, float)
    if len(times) == 0:
        return 0.0, 0.0, 0.0
    yt = y_lin(times)
    df = (1 + yt/2.0) ** (-2.0 * times)
    pv = cfs * df
    dd = times * pv / (1 + yt/2.0)
    dc = pv * times * (times + 0.5) / (1 + yt/2.0)**2
    return float(pv.sum()), float(dd.sum()), float(dc.sum())

def coupon_schedule(maturity, coupon, settle):
    y, mo, d = int(maturity.year), int(maturity.month), int(maturity.day)
    dt = pd.Timestamp(maturity)
    dates = []
    while dt > settle:
        dates.append(dt)
        mo -= 6
        if mo <= 0:
            mo += 12
            y -= 1
        last = calendar.monthrange(y, mo)[1]
        dt = pd.Timestamp(year=y, month=mo, day=min(d, last))
    out = []
    for T in sorted(dates):
        if float(coupon) == 0.0:
            if T == pd.Timestamp(maturity):
                out.append((T, 1.0))
            continue
        amt = float(coupon) / 2.0
        if T == pd.Timestamp(maturity):
            amt += 1.0
        out.append((T, amt))
    return out

liab = pd.DataFrame({
    "date": pd.to_datetime([
        f"{Y}-12-15", f"{Y+1}-06-15", f"{Y+1}-12-15", f"{Y+2}-06-15",
        f"{Y+2}-12-15", f"{Y+3}-06-15", f"{Y+3}-12-15", f"{Y+4}-06-15",
        f"{Y+4}-12-15", f"{Y+5}-06-15", f"{Y+5}-12-15", f"{Y+6}-06-15",
        f"{Y+6}-12-15", f"{Y+7}-06-15", f"{Y+7}-12-15", f"{Y+8}-06-15",
    ]),
    "L": [11, 9, 8, 7, 9, 10, 9, 12, 9, 6, 5, 7, 9, 7, 8, 7],
})
grid = list(liab["date"])
Lvec = liab["L"].to_numpy(float)
K = sum(g <= CUT for g in grid)
n = len(univ)

def next_liability(t):
    for k, g in enumerate(grid):
        if t <= g:
            return k
    return None

F = np.zeros((K, n))          # early cash only
PVt = np.zeros(n); DDt = np.zeros(n); DCt = np.zeros(n)
for j, row in univ.iterrows():
    late_t, late_c = [], []
    for T, amt in coupon_schedule(row["maturity"], row["coupon"], VAL):
        if T <= CUT:
            k = next_liability(T)
            if k is not None and k < K:
                F[k, j] += amt
        else:
            late_t.append((T - VAL).days / 365.25)
            late_c.append(amt)
    PVt[j], DDt[j], DCt[j] = metrics_of_cfs(late_t, late_c)

tail = liab.loc[liab["date"] > CUT]
PV_Lt, DD_Lt, DC_Lt = metrics_of_cfs(
    (tail["date"] - VAL).dt.days.to_numpy() / 365.25,
    tail["L"].to_numpy(),
)
print(f"tail PV={PV_Lt:.4f}  DD={DD_Lt:.4f}  DC={DC_Lt:.4f}")

p = univ["price"].to_numpy(float) / FACE
x = cp.Variable(n, nonneg=True)
s = cp.Variable(K, nonneg=True)
cons = [F[0] @ x - s[0] == Lvec[0]]
for t in range(1, K):
    cons += [F[t] @ x + s[t-1] - s[t] == Lvec[t]]
cons += [PVt @ x == PV_Lt, DDt @ x == DD_Lt, DCt @ x >= DC_Lt]

prob = cp.Problem(cp.Minimize(p @ x), cons)
prob.solve(solver=cp.CLARABEL, verbose=False)
print("status:", prob.status)
print("hybrid cost ($mn):", round(float(prob.value), 4))
print("vs dedication 111.61:", round(111.6088 - float(prob.value), 4))
print("vs immunisation 110.16:", round(float(prob.value) - 110.1578, 4))

xv = np.asarray(x.value).ravel()
idx = np.where(xv > 1e-3)[0]
held = univ.loc[idx].copy()
held["face_$mn"] = xv[idx]
held["cost_$mn"] = p[idx] * xv[idx]
print(held.sort_values("maturity")[["id","type","maturity","coupon","price","face_$mn","cost_$mn"]].to_string(index=False))
print("early surplus:", np.round(np.maximum(np.asarray(s.value).ravel(), 0), 4).tolist())
print("tail PV, DD, DC:",
      round(float(PVt @ xv), 4), round(float(DDt @ xv), 4), round(float(DCt @ xv), 4))


tail PV=62.4359  DD=324.0605  DC=1970.8064
status: optimal
hybrid cost ($mn): 110.8093
vs dedication 111.61: 0.7995
vs immunisation 110.16: 0.6515
          id type   maturity  coupon     price  face_$mn  cost_$mn
N20261115_65 note 2026-11-15  0.0650 100.46875  8.435007  8.474546
N20270615_46 note 2027-06-15  0.0463 100.40625  6.709140  6.736396
N20271115_61 note 2027-11-15  0.0613 102.15625  5.864461  5.990914
N20280615_39 note 2028-06-15  0.0388  99.21875  5.044206  5.004798
N20281115_52 note 2028-11-15  0.0525 101.84375  7.142064  7.273746
N20290430_46 note 2029-04-30  0.0463 100.59375  8.329543  8.379000
N20300831_41 note 2030-08-31  0.0413  98.81250 60.034009 59.321105
B20440515_46 bond 2044-05-15  0.0463  93.68750 10.277532  9.628763
early surplus: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
tail PV, DD, DC: 62.4359 324.0605 2702.6597


### 1.6 Dedication with short sales

We keep the §1.3 cash-balance equalities but let $x_j$ change sign. Buys pay the ask; sales credit the bid,

$$
\min \sum_j \frac{\mathrm{ask}_j}{100}\,x_j^+ - \sum_j \frac{\mathrm{bid}_j}{100}\,x_j^-,
\qquad
x_j = x_j^+ - x_j^-,
\quad x_j^\pm\ge 0,
$$

with the same $Fx+s=L$, $s\ge 0$. Bid–ask is therefore in the objective (typical note spread on this screen is about 3 ticks).

Two model facts then dominate. First, any cash after 15 June 2034 is invisible to $F$, so a short of a 2056 bond books the bid today and never repays principal inside the programme — that is not an arbitrage, it is a missing terminal condition. We therefore drop every issue that matures after the last liability (293 names left). Second, the 0 percent parking rule treats a 8 July 2027 bill and a 15 November 2027 note as perfect substitutes in the December 2027 bucket. Combined with 32nd quotes, that is enough for CBC to declare the LP **unbounded**: there is a long–short overlay with nonnegative parked cash on every grid date and a negative invoice.

A feasible (not optimal) point on that ray already replaces 300 million of November 2027 6.13s with 302 million short July 2027 bills and prints a cost near 102. Put a 5–10 million cap on each CUSIP and the solver harvests hundreds of tiny quote discrepancies and drives the invoice to 89–101. None of that survives a repo special, a position limit, or a requirement that July cash is not the same as November cash.

So: yes, we used the bid–ask spread; no, we did not impose trade-size limits in the textbook LP, and that is why the unrestricted short programme is not a bid. There is no robust risk-free arbitrage in these Barron’s quotes once parking, issue size and financing are respected. The implementable second bid is still the long-only dedicated book at **111.61 million**. Shorts are a basis overlay, not a cheaper dedicated portfolio.

If Clarabel returns unbounded or a wild long–short stack, that is the answer to “did you find arbitrage?”: only inside the 0 percent parking model, not as a trade. Report the long-only 111.61 million as the feasible dedicated bid and use the unbounded flag as the discussion.

In [6]:
import numpy as np
import cvxpy as cp

HORIZON = pd.Timestamp("2034-06-15")
mask = univ["maturity"] <= HORIZON
u = univ.loc[mask].reset_index(drop=True)
# rebuild F on this restricted book (same coupon_schedule / next_liability as §1.3)
n = len(u)
m = len(grid)
F = np.zeros((m, n))
for j, row in u.iterrows():
    for T, amt in coupon_schedule(row["maturity"], row["coupon"], VAL):
        k = next_liability(T)
        if k is not None:
            F[k, j] += amt

p_ask = u["ask"].to_numpy(float) / FACE if "ask" in u.columns else u["price"].to_numpy(float) / FACE
p_bid = u["bid"].to_numpy(float) / FACE if "bid" in u.columns else p_ask - 0.03125/FACE

xp = cp.Variable(n, nonneg=True)     # long face
xm = cp.Variable(n, nonneg=True)     # short face
s  = cp.Variable(m, nonneg=True)
# optional practical cap, e.g. 10.0; leave None / omit for the raw LP
# cons += [xp <= 10, xm <= 10]

cons = [F[0] @ (xp - xm) - s[0] == Lvec[0]]
for t in range(1, m):
    cons += [F[t] @ (xp - xm) + s[t-1] - s[t] == Lvec[t]]

prob = cp.Problem(cp.Minimize(p_ask @ xp - p_bid @ xm), cons)
prob.solve(solver=cp.CLARABEL, verbose=False)
print("status:", prob.status)
print("invoice ($mn):", None if prob.value is None else round(float(prob.value), 4))

xpl = np.asarray(xp.value).ravel() if xp.value is not None else np.zeros(n)
xml = np.asarray(xm.value).ravel() if xm.value is not None else np.zeros(n)
print("n long", int((xpl > 1e-3).sum()), "n short", int((xml > 1e-3).sum()),
      "gross long", round(float(xpl.sum()), 2), "gross short", round(float(xml.sum()), 2))


status: unbounded
invoice ($mn): -inf
n long 0 n short 0 gross long 0.0 gross short 0.0


Allowing shorts, buying at the ask and selling at the bid, the cash-matching LP is unbounded below. No finite dedicated book exists in that formulation. The apparent arbitrage is an artefact of 0 percent inter-date parking (and of dropping post-horizon principal if the long end is left in). Average bid–ask on the screen is about three ticks; we did not impose issue-size or repo constraints. The implementable second bid is therefore the long-only dedicated portfolio from §1.3 at 111.61 million dollars.